In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image, ImageChops, ImageEnhance
from scipy.fftpack import dct

In [ ]:
VALID_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

def list_images(path):
    files = []
    for entry in os.scandir(path):
        if entry.is_file() and entry.name.lower().endswith(VALID_EXT):
            files.append(os.path.join(path, entry.name))
    return files

AUTH_PATHS = [
    "/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2/Au",
    "/kaggle/input/datasets/shriya0/columbia/Columbia Uncompressed Image Splicing Detection/4cam_auth/4cam_auth",
    "/kaggle/input/datasets/saikotahmed/imd2020/IMD2020/Authentic",
]

TAMP_PATHS = [
    "/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2/Tp",
    "/kaggle/input/datasets/shriya0/columbia/Columbia Uncompressed Image Splicing Detection/4cam_splc/4cam_splc",
    "/kaggle/input/datasets/saikotahmed/imd2020/IMD2020/img"
]

In [ ]:
AUTH_FILES = []
TAMP_FILES = []

for p in AUTH_PATHS:
    if os.path.exists(p):
        AUTH_FILES.extend(list_images(p))
    else:
        print("WARNING: Auth path not found →", p)

for p in TAMP_PATHS:
    if os.path.exists(p):
        TAMP_FILES.extend(list_images(p))
    else:
        print("WARNING: Tamp path not found →", p)
all_paths  = AUTH_FILES + TAMP_FILES
all_labels = [0] * len(AUTH_FILES) + [1] * len(TAMP_FILES)

print(f"Authentic images: {len(AUTH_FILES)}")
print(f"Tampered images:  {len(TAMP_FILES)}")
print(f"Total images:     {len(all_paths)} "
      f"(Auth={len(AUTH_FILES)}, Tamp={len(TAMP_FILES)})")

## Tạo datatrain

In [ ]:
import io

def convert_to_ela_image(path, quality=90):
    image = Image.open(path).convert('RGB')

    buffer = io.BytesIO()
    image.save(buffer, 'JPEG', quality=quality)
    buffer.seek(0)
    temp_image = Image.open(buffer).copy()

    ela = ImageChops.difference(image, temp_image)

    extrema = ela.getextrema()
    max_diff = max([ex[1] for ex in extrema])
    max_diff = max_diff if max_diff != 0 else 1

    scale = 255.0 / max_diff
    ela = ImageEnhance.Brightness(ela).enhance(scale)

    return np.array(ela)

In [ ]:
def compute_dct_channel(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = np.float32(gray)
    dct_map = dct(dct(gray.T, norm='ortho').T, norm='ortho')
    dct_map = np.abs(dct_map)
    dct_map = (dct_map - dct_map.min()) / (dct_map.max() - dct_map.min() + 1e-8)
    dct_map = (dct_map * 255).astype(np.uint8)
    return dct_map

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

IMG_SIZE = 224

class CASIADataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]

        # --- Load ELA ---
        ela = convert_to_ela_image(img_path)

        # --- Load RGB for DCT ---
        rgb = cv2.imread(img_path)
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

        dct_map = compute_dct_channel(rgb)

        # --- Resize both ---
        ela = cv2.resize(ela, (IMG_SIZE, IMG_SIZE))
        dct_map = cv2.resize(dct_map, (IMG_SIZE, IMG_SIZE))

        # --- Ensure both are uint8 ---
        ela = ela.astype(np.uint8)
        dct_map = dct_map.astype(np.uint8)

        # --- Apply transforms ONLY to ELA (3 channels) ---
        ela_pil = Image.fromarray(ela)
        if self.transform:
            ela_pil = self.transform(ela_pil)
        else:
            ela_pil = T.ToTensor()(ela_pil)
        
        # --- Convert DCT to tensor without transforms ---
        dct_tensor = torch.tensor(dct_map, dtype=torch.float32).unsqueeze(0) / 255.0

        # --- Merge: ELA(3) + DCT(1) = 4 channels ---
        merged = torch.cat([ela_pil, dct_tensor], dim=0)  # (4, H, W)

        label = torch.tensor(self.labels[idx]).long()
        return merged, label

In [ ]:
from sklearn.model_selection import train_test_split

indices = list(range(len(all_paths)))

idx_temp, idx_test = train_test_split(
    indices, test_size=0.20, random_state=42,
    stratify=all_labels
)

labels_temp = [all_labels[i] for i in idx_temp]
idx_train, idx_val = train_test_split(
    idx_temp, test_size=0.125, random_state=42,
    stratify=labels_temp
)

print(f"Train: {len(idx_train)}  Val: {len(idx_val)}  Test: {len(idx_test)}")

## Train

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torchvision.models as models

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

In [ ]:
import io
from PIL import Image

class JpegCompression:
    def __init__(self, quality=95):
        self.quality = quality

    def __call__(self, img):
        out = io.BytesIO()
        img.save(out, format="JPEG", quality=self.quality)
        out.seek(0)
        return Image.open(out)

In [ ]:
from torchvision import transforms as T

train_transforms = T.Compose([
    # JpegCompression(quality=95),
    # T.RandomResizedCrop(224, scale=(0.85, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.05, contrast=0.05),
    T.ToTensor(),
])

val_transforms = T.Compose([T.ToTensor()])

In [ ]:
BATCH_SIZE = 256

train_dataset = CASIADataset(
    [all_paths[i] for i in idx_train],
    [all_labels[i] for i in idx_train],
    transform=train_transforms
)

val_dataset = CASIADataset(
    [all_paths[i] for i in idx_val],
    [all_labels[i] for i in idx_val],
    transform=val_transforms
)

test_dataset = CASIADataset(
    [all_paths[i] for i in idx_test],
    [all_labels[i] for i in idx_test],
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(
    4, 64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) 

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)
scaler = torch.amp.GradScaler("cuda")

In [ ]:
EPOCHS = 40
EARLY_STOP_PATIENCE = 5
MIN_DELTA = 1e-4

train_loss_list = []
val_loss_list = []
train_acc_list = []
val_acc_list = []

best_val_loss = float("inf")
early_stop_counter = 0
best_state_dict = None

for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss /= len(train_loader)
    train_acc = correct / total

    # VALIDATION
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = correct / total

    # Update LR scheduler theo val_loss
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    # Save history
    train_loss_list.append(train_loss)
    val_loss_list.append(val_loss)
    train_acc_list.append(train_acc)
    val_acc_list.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}]  "
        f"Train Acc: {train_acc:.4f}  Val Acc: {val_acc:.4f}  "
        f"Val Loss: {val_loss:.4f}  LR: {current_lr:.2e}"
    )

    # Early stopping + lưu model tốt nhất
    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        early_stop_counter = 0
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        early_stop_counter += 1
        print(f"EarlyStopping counter: {early_stop_counter}/{EARLY_STOP_PATIENCE}")
        if early_stop_counter >= EARLY_STOP_PATIENCE:
            print("Early stopping triggered.")
            break

# Restore best model theo val_loss
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    model.to(device)
    print(f"Loaded best model with val_loss={best_val_loss:.4f}")

In [ ]:
torch.save(best_state_dict, "best_model.pth")
print("Đã lưu best model vào best_model.pth")

torch.save(model.state_dict(), "model_weights.pth")
print("Đã lưu trọng số model vào model_weights.pth")

checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_val_loss": best_val_loss
}

torch.save(checkpoint, "checkpoint.pth")
print("Đã lưu checkpoint đầy đủ vào checkpoint.pth")

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_loss_list, label='Train Loss')
plt.plot(val_loss_list, label='Val Loss')
plt.title("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_acc_list, label='Train Acc')
plt.plot(val_acc_list, label='Val Acc')
plt.title("Accuracy")
plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import classification_report

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# In báo cáo
print("\n===== CLASSIFICATION REPORT =====\n")
print(classification_report(y_true, y_pred))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Authentic", "Tampered"],
            yticklabels=["Authentic", "Tampered"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Test Set)")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import torch.nn.functional as F

model.eval()
y_true = []
y_prob = []   # dùng để vẽ AUC

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)              # shape [batch, 2]

        probs = F.softmax(outputs, dim=1)    # lấy xác suất
        probs_class1 = probs[:, 1].cpu().numpy()

        y_true.extend(labels.numpy())
        y_prob.extend(probs_class1)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
auc_score = auc(fpr, tpr)

print("AUC:", auc_score)

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()